# Mixture of Experts: Scaling Models with Conditional Computation

## Learning Objectives
1. Understand how gating mechanisms route tokens to sparse experts
2. Implement top-K sparse routing with load balancing
3. Analyze expert specialization and utilization patterns
4. Compare dense vs sparse activation costs and scaling efficiency

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")

## Level 1: Basic MoE Layer

Simplest mixture of experts: all experts process each token with weighted combination

In [ ]:
class BasicMoE(nn.Module):
    """Mixture of Experts with dense routing (all experts used)."""
    
    def __init__(self, dim=64, num_experts=4, expert_hidden=256):
        super().__init__()
        self.dim = dim
        self.num_experts = num_experts
        
        # Gating network: learns to weight each expert
        self.gating = nn.Linear(dim, num_experts)
        
        # Expert networks: independent feedforward networks
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, expert_hidden),
                nn.ReLU(),
                nn.Linear(expert_hidden, dim)
            )
            for _ in range(num_experts)
        ])
    
    def forward(self, x):
        """Args: x (batch, seq_len, dim). Returns: output (batch, seq_len, dim)"""
        gates = torch.softmax(self.gating(x), dim=-1)  # (B, L, E)
        expert_outputs = [expert(x) for expert in self.experts]
        expert_outputs = torch.stack(expert_outputs, dim=-1)  # (B, L, D, E)
        output = torch.einsum('bse,bsde->bsd', gates, expert_outputs)  # (B, L, D)
        return output

# Test basic MoE
batch_size, seq_len, dim = 2, 4, 64
moe_basic = BasicMoE(dim=dim, num_experts=4, expert_hidden=256).to(device)

x = torch.randn(batch_size, seq_len, dim).to(device)
output = moe_basic(x)

total_params = sum(p.numel() for p in moe_basic.parameters())
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Total parameters: {total_params:,}")
print(f"\nBasic MoE: All {moe_basic.num_experts} experts always active")

## Level 2: Sparse MoE with Top-K Routing and Load Balancing

Production-ready implementation: sparse routing, load balancing loss, expert usage tracking

In [ ]:
class SparseMoE(nn.Module):
    """Mixture of Experts with sparse top-K routing and load balancing."""
    
    def __init__(self, dim=64, num_experts=8, expert_hidden=256, k=2, load_balance_weight=0.01):
        super().__init__()
        self.dim = dim
        self.num_experts = num_experts
        self.k = k
        self.load_balance_weight = load_balance_weight
        
        # Gating network: determines routing probabilities
        self.gating = nn.Linear(dim, num_experts)
        
        # Expert networks: independent feedforward networks
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, expert_hidden),
                nn.ReLU(),
                nn.Linear(expert_hidden, dim)
            )
            for _ in range(num_experts)
        ])
    
    def forward(self, x):
        """Sparse routing through top-K experts with load balancing."""
        batch_size, seq_len, dim = x.shape
        
        # Gating: compute softmax probabilities for each expert
        gate_logits = self.gating(x)  # (B, L, E)
        gate_probs = torch.softmax(gate_logits, dim=-1)
        
        # Top-K selection: pick k experts with highest probability
        top_k_gates, top_k_indices = torch.topk(gate_probs, self.k, dim=-1)  # (B, L, K)
        top_k_gates = torch.softmax(top_k_gates, dim=-1)  # renormalize after topk
        
        # Load balancing auxiliary loss: penalize imbalanced expert usage
        expert_freq = gate_probs.mean(dim=(0, 1))  # (E,)
        load_balance_loss = self.num_experts * torch.sum(expert_freq ** 2)
        
        # Compute output: sparse routing through top-K experts
        output = torch.zeros_like(x)
        expert_usage = torch.zeros(self.num_experts).to(device)
        
        for b in range(batch_size):
            for s in range(seq_len):
                for k_idx in range(self.k):
                    expert_idx = top_k_indices[b, s, k_idx].item()
                    gate_weight = top_k_gates[b, s, k_idx]
                    expert_out = self.experts[expert_idx](x[b:b+1, s:s+1])  # (1, 1, D)
                    output[b, s] += gate_weight * expert_out.squeeze()
                    expert_usage[expert_idx] += 1
        
        return output, load_balance_loss, expert_usage

# Test sparse MoE
moe_sparse = SparseMoE(dim=dim, num_experts=8, expert_hidden=256, k=2).to(device)

x = torch.randn(batch_size, seq_len, dim).to(device)
output, loss_balance, usage = moe_sparse(x)

total_params = sum(p.numel() for p in moe_sparse.parameters())
print(f"Output shape: {output.shape}")
print(f"Load balance loss: {loss_balance.item():.4f}")
print(f"Expert usage distribution: {usage.cpu().numpy()}")
print(f"Total parameters: {total_params:,}")
print(f"\nSparse MoE: Only {moe_sparse.k} out of {moe_sparse.num_experts} experts per token")
print(f"Potential speedup: {moe_sparse.num_experts / moe_sparse.k:.1f}x vs dense routing")

## Real-World Example 1: Training MoE with Load Balancing Loss

Complete training loop demonstrating convergence and expert balance

In [ ]:
# Generate synthetic training data
train_size = 100
x_train = torch.randn(train_size, seq_len, dim).to(device)
y_train = torch.randn(train_size, seq_len, dim).to(device)

# Create MoE model for training
moe_model = SparseMoE(dim=dim, num_experts=8, expert_hidden=256, k=2).to(device)
optimizer = optim.Adam(moe_model.parameters(), lr=0.001)

# Training loop with load balancing
train_losses = []
load_balance_losses = []
expert_balance_ratios = []

print("Training MoE model...")
for epoch in range(20):
    optimizer.zero_grad()
    
    # Forward pass
    output, loss_balance, expert_usage = moe_model(x_train)
    
    # Main loss: MSE between output and target
    loss_main = torch.nn.functional.mse_loss(output, y_train)
    
    # Total loss: main + load balancing auxiliary loss
    loss_total = loss_main + 0.01 * loss_balance
    
    # Backward pass and optimization
    loss_total.backward()
    optimizer.step()
    
    # Record metrics
    train_losses.append(loss_main.item())
    load_balance_losses.append(loss_balance.item())
    
    # Calculate expert balance ratio (max_usage / min_usage)
    usage_np = expert_usage.cpu().detach().numpy()
    max_usage = np.max(usage_np) + 1e-6
    min_usage = np.min(usage_np) + 1e-6
    balance_ratio = max_usage / min_usage
    expert_balance_ratios.append(balance_ratio)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/20 | Loss: {loss_main.item():.4f} | "
              f"Load-Balance: {loss_balance.item():.4f} | Balance Ratio: {balance_ratio:.2f}")

print(f"\nTraining complete! Final loss: {train_losses[-1]:.4f}")
print(f"Final balance ratio: {expert_balance_ratios[-1]:.2f} (1.0 = perfect balance)")

## Real-World Example 2: Analyzing Expert Specialization and Routing Patterns

Monitor which experts activate for different inputs and sequence positions

In [ ]:
# Analyze expert routing patterns
moe_model.eval()

# Generate test data
test_size = 50
x_test = torch.randn(test_size, seq_len, dim).to(device)

# Track expert usage by position
expert_usage_by_position = defaultdict(lambda: torch.zeros(moe_model.num_experts))

with torch.no_grad():
    for i in range(test_size):
        x_sample = x_test[i:i+1]  # (1, seq_len, dim)
        gate_logits = moe_model.gating(x_sample)
        gate_probs = torch.softmax(gate_logits, dim=-1)  # (1, L, E)
        top_k_gates, top_k_indices = torch.topk(gate_probs, moe_model.k, dim=-1)
        
        # Track which experts are selected for each position
        for pos in range(seq_len):
            for k_idx in range(moe_model.k):
                expert_idx = top_k_indices[0, pos, k_idx].item()
                expert_usage_by_position[pos][expert_idx] += 1

# Display specialization patterns
print("Expert Usage by Sequence Position:")
print("-" * 60)
for pos in range(seq_len):
    usage = expert_usage_by_position[pos]
    top_expert = usage.argmax().item()
    top_usage = usage[top_expert].item()
    total_usage = usage.sum().item()
    print(f"Position {pos}: Expert {top_expert} selected {top_usage:.0f}/{total_usage:.0f} times "
            f"({100*top_usage/total_usage:.1f}%)")

print("\nObservation: Different positions may prefer different experts")
print("This demonstrates emergent specialization in sparse MoE models!")

## Real-World Example 3: Dense vs Sparse Efficiency Comparison

Benchmark computational costs as number of experts increases

In [ ]:
# Create comprehensive visualization of training and efficiency
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Main loss convergence
axes[0, 0].plot(train_losses, marker='o', linewidth=2, color='blue')
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('MSE Loss', fontsize=11)
axes[0, 0].set_title('MoE Training Loss Convergence', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Load balancing loss
axes[0, 1].plot(load_balance_losses, marker='s', linewidth=2, color='orange')
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Load Balance Loss', fontsize=11)
axes[0, 1].set_title('Auxiliary Load Balancing Loss', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Expert balance ratio
axes[1, 0].plot(expert_balance_ratios, marker='^', linewidth=2, color='green')
axes[1, 0].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect balance')
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Max Usage / Min Usage', fontsize=11)
axes[1, 0].set_title('Expert Utilization Balance', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Plot 4: Computation cost comparison (sparse vs dense)
x_pos = np.arange(len(num_experts_list))
width = 0.25

# Normalize to dense cost = 1.0
dense_normalized = [1.0] * len(num_experts_list)
sparse_k1_normalized = [t / d for t, d in zip(sparse_times_k1, dense_times)]
sparse_k2_normalized = [t / d for t, d in zip(sparse_times_k2, dense_times)]

axes[1, 1].bar(x_pos - width, dense_normalized, width, label='Dense', color='blue', alpha=0.7)
axes[1, 1].bar(x_pos, sparse_k2_normalized, width, label='Sparse (k=2)', color='green', alpha=0.7)
axes[1, 1].bar(x_pos + width, sparse_k1_normalized, width, label='Sparse (k=1)', color='orange', alpha=0.7)

axes[1, 1].set_xlabel('Number of Experts', fontsize=11)
axes[1, 1].set_ylabel('Relative Computation Time', fontsize=11)
axes[1, 1].set_title('Dense vs Sparse Routing Cost', fontsize=12, fontweight='bold')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(num_experts_list)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/moe_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nMoE analysis and visualization complete!")
print(f"Key finding: With {num_experts_list[-1]} experts, sparse routing is ~{1/(sparse_times_k2[-1]/dense_times[-1]):.1f}x faster than dense")